# Lab 3: How much rain falls on a rainy day, and how likely is a very heavy one?

**Estimated time:** about 75 minutes

**Difficulty:** mathematics: Developing; coding: Introductory to Developing (run and modify); science concepts: Developing; interpretation: Moderate

## What you will be able to do

1. Separate rainfall into two questions, whether it rains and how much falls when it does, and measure each from real data.
2. Explain why a bell curve is the wrong model for daily rainfall, using a number from the data.
3. Fit a gamma distribution to wet-day rainfall and judge where it works and where it fails, especially for heavy days.
4. Convert a probability per wet day into an expected number of heavy days per year, and check that the pieces add up to the annual total.

## What this lab assumes

- Lab 2 (mean, standard deviation, skewness, histograms).
- The idea that probability is the area under a density curve (the same area-under-a-curve idea as accumulated rainfall in Lab 1).

## How this lab works

Each part follows the same pattern: **predict** what you expect, **calculate**
it, **visualize** it, then **explain** what you found. Write your prediction
before you run the code. Being wrong is useful information here; it is the
reason for the prediction step.

**Before you run the first cell:** Colab will show a popup warning that this
notebook "was not authored by Google." Colab shows this for every notebook
opened from GitHub, not just this one, and it does not mean anything is wrong.
Click "Run anyway" to continue.

## Step 0: Set up

In [ ]:
# Install the course library from GitHub. The @v0.3.0 pins this notebook to
# one tagged release, so everyone gets exactly the same code and data.
# Colab starts a fresh environment every session, so this line runs every
# time you open the notebook. That is normal, not an error.

!pip install git+https://github.com/SaharMK22/qi_course.git@v0.3.0 --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import qi_course as qc

# Confirm which version of the library is running.
print("qi_course version:", qc.__version__)

## Step 1: The scientific question

Daily rainfall is not like daily temperature. Many days have none at all,
most rainy days bring a modest amount, and a few bring a great deal. Planning
for floods, sizing storm drains, and judging drought all depend on how that
pattern looks.

> If you pick a day at random, how likely is it to rain? If it rains, how much
> should you expect? And how likely is a day heavy enough to cause trouble?

You will use the same real station records as Lab 2, this time with the
daily precipitation column.

In [ ]:
# Choose a station (any piece of a name from qc.list_stations()), and decide
# what counts as a "wet day". 1 mm is a common choice: it ignores trace
# amounts too small to matter.
STATION = "New Orleans"
WET_MM = 1.0

## Step 2: Predict (before running any more code)

Write rough answers in the cell below.

1. What fraction of days at this station have at least 1 mm of rain? One in ten,
   one in four, one in two?
2. On a day that does rain, what is a typical amount? (1 mm is a drizzle,
   10 mm is a steady rain, 50 mm is about two inches.)
3. Imagine a histogram of the rainfall on **every** day, including dry ones.
   Which bar is tallest? Sketch it.
4. A bell curve could in principle be drawn for any data. What would be
   physically impossible about a bell curve for daily rainfall?
5. How often do you expect a day with more than 50 mm: every year, every ten
   years, almost never?

*Double-click this cell and type your predictions here:*

## Step 3: Look at the data

Same record format as Lab 2: one row per day, `prcp_mm` in millimeters, NaN for
missing days. Two details specific to rain: amounts were measured in
hundredths of an inch, so they come in steps of about 0.25 mm; and trace
amounts too small to measure appear as zero.

In [ ]:
df = qc.load_station(STATION)
print(df.attrs["label"])

coverage = qc.station_coverage(df)
print("Record runs from", coverage["first_year"], "to", coverage["last_year"])
print("Percent of days with a rainfall value:", coverage["pct_present"]["prcp_mm"])

rain = df["prcp_mm"].dropna()     # every day that has a trustworthy value
print("\nWettest day on record:", rain.max(), "mm")

## Step 4: Calculate, in two pieces

Rainfall is really two questions. **Occurrence:** does it rain? **Intensity:**
if it does, how much? We measure each separately.

In [ ]:
summary = qc.wet_day_summary(rain, wet_threshold_mm=WET_MM)
wet = summary["wet_amounts"]                  # rainfall on wet days only

print(f"Days with a record:          {summary['n_days']}")
print(f"Occurrence: wet-day fraction {summary['wet_fraction']:.1%}  (about {summary['wet_fraction']*365.25:.0f} wet days a year)")
print(f"Intensity:  mean wet-day amount {summary['mean_wet_mm']:.1f} mm, std {summary['std_wet_mm']:.1f} mm")
print(f"            skewness {summary['skewness_wet']:.2f}")

Compare with your predictions. Then a check on whether the two pieces
describe the whole record. If the pieces are right, then

**average annual total = wet days per year x average wet-day amount**

In [ ]:
wet_days_per_year = summary["wet_fraction"] * 365.25
predicted_total = wet_days_per_year * summary["mean_wet_mm"]

totals = qc.annual_totals(df)       # one total per year, using only nearly complete years
actual_total = totals.mean()

print(f"Built from the two pieces: {predicted_total:.0f} mm per year")
print(f"Measured annual average:   {actual_total:.0f} mm per year")
print(f"Difference: {100*(predicted_total-actual_total)/actual_total:+.1f}%")

## Step 5: Why not a bell curve?

Fit a bell curve to the wet-day amounts and ask how much of it falls below
zero, where rain cannot go.

In [ ]:
mean_w, std_w = summary["mean_wet_mm"], summary["std_wet_mm"]

# Area of a bell curve below zero: the standard normal CDF at z.
z = (0 - mean_w) / std_w
p_negative = 0.5 * (1 + math.erf(z / math.sqrt(2)))
print(f"A bell curve with this mean and spread puts {p_negative:.1%} of wet days at NEGATIVE rainfall.")

## Step 6: A better model, the gamma distribution

The gamma distribution cannot go below zero, can pile up near zero, and has a
long tail to the right. Two numbers describe it, and both come from the mean
and variance you already computed:

shape = mean squared / variance      scale = variance / mean

In [ ]:
grid, pdf, params = qc.fit_gamma_pdf(wet)
print(f"Fitted gamma: shape = {params['shape']:.2f}, scale = {params['scale']:.1f} mm")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
bins = np.arange(0, np.ceil(wet.max()) + 5, 5)           # bins 5 mm wide

# Left: zoomed in on ordinary wet days.
qc.histogram_with_fit(wet, grid, pdf, bins=bins, fit_label="Fitted gamma",
                      title="Wet-day rainfall, up to 60 mm", xlabel="Rainfall on a wet day (mm)", ax=axes[0])
axes[0].set_xlim(0, 60)

# Right: the whole range, with a logarithmic vertical axis so the rare big days are visible.
qc.histogram_with_fit(wet, grid, pdf, bins=bins, fit_label="Fitted gamma",
                      title="Same data, log scale", xlabel="Rainfall on a wet day (mm)", ax=axes[1])
axes[1].set_yscale("log")
plt.tight_layout()
plt.show()

## Step 7: How likely is a heavy day?

The probability that a wet day exceeds some amount is the **area under the
density to the right of that amount**, the same area-under-a-curve idea as
accumulated rainfall in Lab 1. The library computes that area with the
trapezoid rule. Compare it with simply counting days in the record.

In [ ]:
rows = []
for threshold in [10, 25, 50, 100, 150]:
    observed = qc.empirical_exceedance(wet, threshold)                              # count days
    modeled = qc.gamma_exceedance(threshold, params["shape"], params["scale"])      # area under curve
    per_year = observed * wet_days_per_year                                          # heavy days per year
    rows.append({
        "threshold_mm": threshold,
        "observed P(exceed | wet day)": observed,
        "gamma P(exceed | wet day)": modeled,
        "observed days per year": per_year,
        "about once every": (f"{1 / per_year:.1f} years" if 0 < per_year < 1 else
                             ("more than once a year" if per_year >= 1 else "never in this record")),
    })
pd.DataFrame(rows).round(4)

Where do the gamma and the observed counts agree, and where do they drift
apart? Which direction do they drift: does the model overstate or understate
the heaviest days? For a storm-drain designer, which mistake is worse?

## Step 8: Totals behave differently from days

Daily rainfall is strongly skewed. What about the **annual total**, which adds
up a few hundred of those skewed days?

In [ ]:
totals = qc.annual_totals(df)
tstats = qc.descriptive_stats(totals)
print(f"Years used: {tstats['n']}")
print(f"Mean annual total {tstats['mean']:.0f} mm, std {tstats['std']:.0f} mm")
print(f"Skewness of daily wet-day rain: {summary['skewness_wet']:.2f}")
print(f"Skewness of annual totals:      {tstats['skewness']:.2f}")

grid_t, pdf_t, _ = qc.fit_normal_pdf(totals)
qc.histogram_with_fit(totals, grid_t, pdf_t, bins=12, title="Annual total rainfall vs. a bell curve",
                      xlabel="Annual total (mm)")
plt.show()

## Step 9: Explain

1. **Two questions.** Which was easier to predict, occurrence or intensity? Which
   mattered more to your annual total? How did your Step 2 predictions compare?
2. **Not a bell curve.** Use the negative-rainfall number to explain in two
   sentences why a bell curve fails for daily rain, even though it worked for July highs in Lab 2.
3. **The tail.** Using the table in Step 7, say whether the gamma is safe to use
   for deciding how big a storm drain should be, and why.
4. **Sums versus days.** Annual totals are much less skewed than daily amounts.
   What does adding many skewed days together do to the shape? Is the annual total
   closer to a bell curve than the days are? Is it exactly one?
5. **Wet-day threshold.** Change `WET_MM` at the top to 0.1 and then to 5 and re-run.
   What happens to the wet-day fraction and to the mean wet-day amount, and why?

## Optional extension: Use your own city

The same search used in Lab 2 works here. Replace the `#` lines below, choose a
station with a 100-year record near your city, set `STATION` to its id, and re-run
from Step 3. Read the coverage report before trusting the result.

In [ ]:
# nearby = qc.find_stations(lat=29.76, lon=-95.37, n=8, min_years=100)   # Houston, as an example
# nearby

## If something breaks

The install cell is the one most likely to act up. If a later cell reports
that a function does not exist, use Runtime > Restart session, then run the
cells again from the top. For any other error, do not guess. Copy the exact
error text, note which cell it came from, and send both to your instructor or TA.

---

## Notes for the instructor (draft; delete this cell before sharing with students)

**Status:** first working draft using real NOAA GHCN-Daily precipitation. Time estimates are mine and untested with students.

**What students should find** (wet day = at least 1 mm; gamma fitted to wet-day amounts by moments):

| Station | Wet-day fraction | Mean wet-day amount (mm) | Skewness of wet-day rain | Gamma shape | Observed P(>50 mm given wet) | Gamma P(>50 mm given wet) | Mean annual total (mm) | Skewness of annual totals |
|---|---|---|---|---|---|---|---|---|
| Los Angeles (downtown) | 8% | 13.2 | 2.70 | 0.72 | 3.73% | 3.61% | 368 | 0.70 |
| New Orleans (Audubon) | 26% | 16.5 | 3.84 | 0.59 | 6.39% | 7.51% | 1575 | 0.63 |
| Portland, OR (Bull Run headworks) | 44% | 12.5 | 2.47 | 1.03 | 1.84% | 1.73% | 2014 | 0.69 |
| San Francisco (downtown) | 14% | 10.3 | 2.75 | 0.86 | 1.04% | 1.09% | 537 | 0.53 |

**Intended discoveries:** (1) occurrence and intensity multiply to the annual total (Step 4 check); (2) a bell curve puts a large share of probability below zero; (3) a gamma fits the bulk well but its tail can be off in either direction depending on the station; (4) sums of skewed days are far less skewed (central limit idea) but still not perfectly Gaussian.

**Likely misconceptions:** treating dry days as missing data (they are real zeros); confusing probability per wet day with probability per day; reading a "once every N years" figure as a schedule rather than an average; assuming a good fit in the middle means a good fit in the tail.

**Honest limitations:** method-of-moments is chosen for transparency, not accuracy; maximum likelihood fits would differ somewhat. The gamma has no memory of storm type (a hurricane and a thunderstorm are mixed together). Gauge undercatch and trace handling affect small amounts. A 100-year record cannot reliably estimate 100-year events, so the largest rows of the Step 7 table rest on very few days. Annual totals use only years with at least 95 percent of days present.

**Open questions for Dale:**
1. Is gamma the right distribution to introduce, or would an exponential (one parameter) be a better first step?
2. Should extreme-value statistics (annual maxima) come here or in a later lab?
3. Should the Houston and Harvey storyline from Lab 1 connect to this lab through a Houston station?